# Leçon 03 — Les arbres binaires
*Structures hiérarchiques — représentation par une liste de dictionnaires*

Ce cours prépare les exercices 01 à 06 de la feuille. Il suit l'ordre de la feuille :
1. le besoin et le vocabulaire ;
2. lire et écrire un arbre stocké dans une liste ;
3. les fonctions récursives sur un arbre (somme, taille, hauteur) ;
4. les parcours en profondeur (préfixe, infixe, postfixe) ;
5. le parcours en largeur, avec une file ;
6. la fiche réflexe pour le contrôle.

## 1. Le besoin

Les structures vues jusqu'ici (listes, piles, files) sont **linéaires** : chaque élément a au plus **un** suivant. Elles rangent des choses « en file indienne ».

Beaucoup d'informations ne sont pas rangées comme ça, mais **en hiérarchie** : un élément est au-dessus de plusieurs autres.
- Un arbre généalogique : une personne a des enfants, qui ont eux-mêmes des enfants.
- Les dossiers de l'ordinateur : un dossier contient des sous-dossiers, qui en contiennent d'autres.
- Un tournoi : la finale vient de deux demi-finales, chacune vient de deux quarts de finale.

Pour représenter cette hiérarchie, on utilise un **arbre**. Dans un **arbre binaire**, chaque élément a **au plus deux** éléments en dessous de lui : un **à gauche** et un **à droite**.

## 2. Le vocabulaire

On prend l'arbre 1 de l'exercice 02 :

```
                A
             /     \
           B         F
          / \       / \
         C   D     G   H
          \       /     \
           E     I       J
```

| Mot | Définition | Exemple dans l'arbre 1 |
|:---|:---|:---|
| **nœud** | un élément de l'arbre ; il porte une **valeur** (on dit aussi **étiquette**) | A, B, …, J sont les 10 nœuds |
| **racine** | le seul nœud qui n'a personne au-dessus de lui ; il est dessiné en haut | A |
| **fils gauche / fils droit** | le nœud placé juste en dessous, à gauche / à droite | le fils gauche de B est C, son fils droit est D |
| **père** | le nœud placé juste au-dessus ; chaque nœud sauf la racine a **un seul** père | le père de E est C |
| **feuille** | un nœud qui n'a **aucun** fils | D, E, I, J |
| **sous-arbre** | un nœud **avec tout ce qui est en dessous de lui** | le sous-arbre gauche de A est celui de racine B : {B, C, D, E} |
| **arbre vide** | un arbre qui n'a aucun nœud | sous C à gauche, il n'y a rien : c'est un sous-arbre vide |
| **taille** | le nombre de nœuds | 10 |
| **hauteur** | le nombre de **niveaux** | 4 (A / B F / C D G H / E I J) |

⚠️ **Attention à la hauteur.** Sur la feuille, la convention est : **arbre vide → 0, une feuille seule → 1**. On compte donc les niveaux (les nœuds sur le plus long chemin de la racine à une feuille).
Certains sujets de bac utilisent une autre convention (une feuille seule → 0, on compte les **arêtes**, les traits). Le jour du contrôle, **lis toujours la convention donnée dans l'énoncé**.

**Gauche et droite comptent.** Un nœud qui a un seul fils peut l'avoir à gauche ou à droite, et ce ne sont pas les mêmes arbres. Dans l'arbre 1, E est le fils **droit** de C (C n'a pas de fils gauche), et I est le fils **gauche** de G.

## 3. Stocker un arbre dans une liste

### 3.1 Le besoin

Python n'a pas de type « arbre ». Il faut donc ranger l'arbre avec ce qu'on connaît déjà : des **listes** et des **dictionnaires**.

### 3.2 L'idée

- Chaque nœud est un **dictionnaire** avec trois clés :
  - `'v'` : la **valeur** du nœud ;
  - `'g'` : l'**indice**, dans la liste, de son fils gauche ;
  - `'d'` : l'**indice**, dans la liste, de son fils droit.
- Tous les nœuds sont rangés dans **une liste**. La **racine est à l'indice 0**.
- S'il n'y a pas de fils, on écrit **`-1`** : c'est un indice qui ne désigne aucune case ; il veut dire « ici, le sous-arbre est **vide** ».

Un `'g'` ou un `'d'` n'est **pas** une valeur : c'est une **adresse** (un numéro de case). Pour trouver le fils, on va voir **la case** de la liste qui porte ce numéro.

In [1]:
petit = [{'v':'A', 'g': 1, 'd': 2},    # case 0 : A, fils gauche en case 1, fils droit en case 2
         {'v':'B', 'g':-1, 'd':-1},    # case 1 : B, aucun fils -> feuille
         {'v':'C', 'g':-1, 'd':-1}]    # case 2 : C, aucun fils -> feuille

racine = petit[0]
print(racine['v'])                    # la valeur de la racine
print(petit[racine['g']]['v'])        # on suit l'indice 'g' : case 1
print(petit[racine['d']]['v'])        # on suit l'indice 'd' : case 2

A
B
C


Cet arbre est :
```
    A
   / \
  B   C
```

### 3.3 Méthode : passer de la liste au dessin (exercice 01)

1. Numérote chaque dictionnaire de la liste : 0, 1, 2… (écris les numéros au crayon au-dessus).
2. Dessine la racine : c'est la case **0**.
3. Pour chaque nœud dessiné, lis `'g'` et `'d'` :
   - si c'est un nombre, va à cette case et dessine sa valeur **en dessous, du bon côté** ;
   - si c'est `-1`, ne dessine rien de ce côté.
4. Recommence jusqu'à ce que tous les nœuds aient leurs fils.

⚠️ L'ordre des cases dans la liste n'a **rien à voir** avec la forme du dessin : seuls `'g'` et `'d'` décident de qui est sous qui.

### 3.4 Méthode : passer du dessin à la liste (exercice 02)

1. Numérote les nœuds sur le dessin. La racine **doit** avoir le numéro 0 (les fonctions commencent à `i=0`). Pour les autres, n'importe quel ordre marche ; le plus simple est de numéroter **niveau par niveau, de gauche à droite**.
2. Pour chaque nœud, écris un dictionnaire dans la case de son numéro : sa valeur, le numéro de son fils gauche (ou `-1`), le numéro de son fils droit (ou `-1`).
3. Vérifie : chaque numéro sauf 0 apparaît **exactement une fois** dans un `'g'` ou un `'d'` (chaque nœud a un seul père).

### 3.5 Un outil pour vérifier ton dessin

La prof a une fonction `afficher_arbre` qui fait un vrai dessin. Ici, on écrit un petit outil qui dessine l'arbre **couché** (la racine à gauche). Ce n'est pas à apprendre : il sert seulement à vérifier.

In [2]:
# Outil (pas à apprendre) : dessine l'arbre "couché", la racine à gauche.
# Le fils droit est au-dessus, le fils gauche en dessous :
# penche la tête vers la gauche pour le voir dans le bon sens.
def dessiner(arbre, i=0, decalage=""):
    if i == -1:
        return
    noeud = arbre[i]
    dessiner(arbre, noeud['d'], decalage + "      ")
    print(decalage + str(noeud['v']))
    dessiner(arbre, noeud['g'], decalage + "      ")

dessiner(petit)

      C
A
      B


## 4. Les fonctions récursives sur un arbre

### 4.1 Le besoin

On veut calculer quelque chose sur **tout** l'arbre : la somme des valeurs, le nombre de nœuds, la hauteur…

Une boucle `for` sur la liste suffirait pour la somme ou la taille, mais pas pour la hauteur ni pour les parcours : il faut suivre la **forme** de l'arbre.

L'observation qui débloque tout : **un arbre, c'est une racine + un sous-arbre gauche + un sous-arbre droit**, et ces sous-arbres sont eux-mêmes des arbres. Une définition qui se sert d'elle-même : c'est exactement le cas où on utilise la **récursivité**.

### 4.2 Le modèle (à connaître par cœur)

Toutes les fonctions des exercices 03, 04 et 06 ont **la même forme** :

```python
def f(arbre, i=0):
    if i == -1:                 # CAS DE BASE : le sous-arbre est vide
        return ...              # la réponse pour un arbre vide
    noeud = arbre[i]            # le dictionnaire du nœud
    ... f(arbre, noeud['g']) ...   # on demande la réponse au sous-arbre gauche
    ... f(arbre, noeud['d']) ...   # on demande la réponse au sous-arbre droit
    return ...                  # on combine avec la valeur du nœud
```

- Le paramètre `i` est l'**indice de la racine du sous-arbre** qu'on traite. `i=0` par défaut : si on ne donne rien, on traite l'arbre entier.
- Le **cas de base** est l'arbre **vide** (`i == -1`), pas la feuille. Une feuille est traitée comme les autres nœuds : ses deux appels tombent sur `-1` et renvoient la réponse du vide.
- Les deux seules questions à se poser :
  1. **Que vaut la réponse pour un arbre vide ?**
  2. **Comment combiner** la valeur du nœud, la réponse de gauche et la réponse de droite ?

| Fonction | Arbre vide | Combinaison |
|:---|:---|:---|
| somme | `0` | `valeur + somme_gauche + somme_droite` |
| taille | `0` | `1 + taille_gauche + taille_droite` |
| hauteur | `0` | `1 + le plus grand des deux` |
| liste des feuilles | `[]` | `feuilles_gauche + feuilles_droite` (et `[valeur]` pour une feuille) |
| préfixe | `[]` | `[valeur] + gauche + droite` |
| infixe | `[]` | `gauche + [valeur] + droite` |
| postfixe | `[]` | `gauche + droite + [valeur]` |

### 4.3 L'exemple de la feuille : la somme

In [3]:
def somme_liste(arbre, i=0):
    if i == -1:                        # cas de base : sous-arbre vide
        return 0
    noeud = arbre[i]                   # le dictionnaire du nœud
    return noeud['v'] + somme_liste(arbre, noeud['g']) + somme_liste(arbre, noeud['d'])

nombres = [{'v':5, 'g':1, 'd':2}, {'v':3, 'g':-1, 'd':-1},
           {'v':10, 'g':3, 'd':-1}, {'v':2, 'g':-1, 'd':-1}]
dessiner(nombres)
print(somme_liste(nombres))

      10
            2
5
      3
20


### 4.4 Méthode papier : l'arbre des appels

Pour comprendre (ou vérifier) une fonction récursive sur papier, on dessine **l'arbre des appels** : chaque appel écrit ses deux appels en dessous, et on remonte les résultats.

On note `s(i)` pour `somme_liste(nombres, i)` :

```
                         s(0) = 5 + 3 + 12 = 20
                 /                              \
     s(1) = 3 + 0 + 0 = 3                s(2) = 10 + 2 + 0 = 12
        /         \                         /              \
   s(-1)=0     s(-1)=0         s(3) = 2 + 0 + 0 = 2      s(-1)=0
                                  /         \
                             s(-1)=0     s(-1)=0
```

On compte **9 appels** : 4 sur des nœuds et 5 sur des sous-arbres vides.
Règle générale : pour un arbre de `n` nœuds, il y a toujours `n + 1` appels sur le vide, donc `2n + 1` appels en tout (ici `2 × 4 + 1 = 9`).

### 4.5 Une fonction qui renvoie une liste : les feuilles

On peut aussi **construire une liste**. On utilise la **concaténation** : `[1, 2] + [3] + []` donne `[1, 2, 3]`.

Ici, il y a **deux** situations particulières : l'arbre vide (on renvoie `[]`) et la feuille (on renvoie `[sa valeur]`).

In [4]:
def feuilles_liste(arbre, i=0):
    if i == -1:                                     # sous-arbre vide : aucune feuille
        return []
    noeud = arbre[i]
    if noeud['g'] == -1 and noeud['d'] == -1:       # le nœud est une feuille
        return [noeud['v']]
    return feuilles_liste(arbre, noeud['g']) + feuilles_liste(arbre, noeud['d'])

print(feuilles_liste(nombres))

[3, 2]


## 5. Les parcours en profondeur

### 5.1 Le besoin

On veut **visiter tous les nœuds** et écrire leurs valeurs dans une liste, par exemple pour les afficher ou pour chercher une valeur. Mais dans quel ordre ? Il y a plusieurs ordres « naturels ».

Dans un parcours **en profondeur**, on descend **jusqu'en bas d'un côté** avant de s'occuper de l'autre côté. On traite toujours le sous-arbre gauche avant le sous-arbre droit. Ce qui change, c'est **le moment où l'on écrit la valeur du nœud** :

| Parcours | Ordre | Moyen mnémotechnique |
|:---|:---|:---|
| **préfixe** | nœud, **puis** gauche, **puis** droite | « **pré** » = **avant** : on écrit le nœud **avant** ses fils |
| **infixe** | gauche, **puis** nœud, **puis** droite | « **in** » = **au milieu** |
| **postfixe** | gauche, **puis** droite, **puis** nœud | « **post** » = **après** : on écrit le nœud **après** ses fils |

### 5.2 Méthode papier : le tour de l'arbre

Dessine un trait qui fait le tour de l'arbre en partant de la gauche de la racine, en collant aux nœuds (comme si tu suivais le bord avec ton doigt). Chaque nœud est longé **trois fois** : par la gauche, par en dessous, par la droite.
- **préfixe** : on écrit le nœud quand on passe **à sa gauche** (1ʳᵉ fois) ;
- **infixe** : quand on passe **en dessous** (2ᵉ fois) ;
- **postfixe** : quand on passe **à sa droite** (3ᵉ fois, en remontant).

Sur l'arbre 1 :
- préfixe : A B C E D F G I H J
- infixe : C E B D A I G F H J
- postfixe : E C D B I G J H F A

Vérifications rapides : en préfixe, la **racine est en premier** ; en postfixe, la **racine est en dernier** ; en infixe, la racine **sépare** les nœuds de gauche de ceux de droite.

## 6. Le parcours en largeur

### 6.1 Le besoin

Parfois on veut lire l'arbre **niveau par niveau**, de haut en bas et, dans chaque niveau, de gauche à droite. Sur l'arbre 1 : A, puis B F, puis C D G H, puis E I J.

La récursivité ne marche pas bien ici : elle descend jusqu'en bas d'un côté avant de passer à l'autre, alors qu'on veut finir tout un niveau avant de descendre.

### 6.2 L'idée : une file

On utilise une **file** (premier entré, premier sorti, leçon 02E) pour ranger **les nœuds qu'il reste à visiter**, dans l'ordre où on les découvre.

1. On enfile l'indice de la racine.
2. Tant que la file n'est pas vide :
   - on défile un indice, on écrit la valeur de ce nœud ;
   - on enfile l'indice de son fils gauche, puis celui de son fils droit (seulement s'ils ne valent pas `-1`).

Comme la file rend les éléments **dans l'ordre d'arrivée**, tous les nœuds d'un niveau sortent avant ceux du niveau suivant (ils ont été enfilés avant).

### 6.3 Méthode papier : le tableau de la file

| Défilé | Écrit | On enfile | File après (tête à gauche) |
|:---|:---|:---|:---|
| | | 0 (A) | A |
| A | A | B, F | B F |
| B | A B | C, D | F C D |
| F | A B F | G, H | C D G H |
| C | A B F C | E | D G H E |
| D | … D | (rien) | G H E |
| G | … G | I | H E I |
| H | … H | J | E I J |
| E, I, J | … E I J | (rien) | (vide) |

Résultat : A B F C D G H E I J.

## 7. Fiche réflexe pour le contrôle

1. **Lire une liste** → numérote les cases, racine = case 0, suis `'g'` et `'d'`, `-1` = rien.
2. **Écrire une liste** → numérote le dessin (racine = 0), un dictionnaire par nœud, `-1` quand il n'y a pas de fils.
3. **Fonction récursive** → `if i == -1: return (réponse du vide)`, puis `noeud = arbre[i]`, puis deux appels avec `noeud['g']` et `noeud['d']`, puis on combine.
4. **Hauteur** → vide 0, sinon `1 +` le plus grand des deux (utilise `max(a, b)` ou un `if`). Vérifie la convention de l'énoncé.
5. **Préfixe / infixe / postfixe** → seule la place de `[noeud['v']]` change dans la concaténation.
6. **Largeur** → pas de récursivité : une file d'indices, `while not F.est_vide()`, on enfile gauche puis droite si ce n'est pas `-1`.
7. **Sur papier** → arbre des appels pour une fonction récursive, tour de l'arbre pour les profondeurs, tableau de la file pour la largeur.